In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
if os.path.exists("/content/commitgen"):
    !cd /content/commitgen && git pull
else:
    !git clone https://github.com/Nilay-Mehta/commitgen.git /content/commitgen
%cd /content/commitgen

In [ ]:
%pip install -q -r requirements.txt
%pip install -q gguf protobuf sentencepiece

In [ ]:
ADAPTER = "/content/drive/MyDrive/commitgen_checkpoints/qwen25coder-lora-full-0/checkpoint-2000"
MERGED_DIR = "/content/commitgen/deployment/merged_model"
!python deployment/merge_lora.py --adapter {ADAPTER} --output {MERGED_DIR}

In [ ]:
# Clone llama.cpp and build the quantize binary. CMake is the modern path.
# Takes ~3-5 minutes on Colab CPU.
import os
if not os.path.exists("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF 2>&1 | tail -5
!cd /content/llama.cpp && cmake --build build --config Release -j --target llama-quantize 2>&1 | tail -5

In [ ]:
# Convert merged HF model to fp16 GGUF
MERGED_DIR = "/content/commitgen/deployment/merged_model"
FP16_GGUF = "/content/commitgen/deployment/commitgen-fp16.gguf"
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {FP16_GGUF} --outtype f16
!ls -lh {FP16_GGUF}

In [ ]:
# Quantize fp16 -> Q4_K_M (~350 MB)
FP16_GGUF = "/content/commitgen/deployment/commitgen-fp16.gguf"
Q4_GGUF = "/content/commitgen/deployment/commitgen-Q4_K_M.gguf"
!/content/llama.cpp/build/bin/llama-quantize {FP16_GGUF} {Q4_GGUF} Q4_K_M
!ls -lh {Q4_GGUF}

In [ ]:
# Copy final GGUF + Modelfile to Drive so we can download them
DEPLOY_DRIVE = "/content/drive/MyDrive/commitgen_deployment"
!mkdir -p {DEPLOY_DRIVE}
!cp /content/commitgen/deployment/commitgen-Q4_K_M.gguf {DEPLOY_DRIVE}/
!cp /content/commitgen/deployment/Modelfile {DEPLOY_DRIVE}/
!ls -lh {DEPLOY_DRIVE}/